# VCuboidFIT trên Google Colab: Tầng 0 + Tầng 1 (và chế độ agent)
Chạy lần lượt từ trên xuống; mọi bước **chạy lại an toàn**. Chọn runtime **T4 GPU** (không dùng P100) cho Tầng 1; Tầng 0 chạy được không cần GPU. Dùng Kaggle thì mở `run_on_kaggle.ipynb`.

| Phần | Việc |
|---|---|
| 1 | Nạp dữ liệu nuScenes: **từ Drive** (`MyDrive/vcf_upload`), hoặc dán link tải, hoặc upload tay |
| 2-4 | Cấu hình, kiểm GPU, cài môi trường (Drive chỉ để lưu kết quả + cache, tuỳ chọn `USE_DRIVE`) |
| 5-7 | Tầng 0 → Tầng 1 → Tầng 0 lần hai + báo cáo |
| 8 | Lưu kết quả (Drive hoặc `/content`) |
| 9 | (Tuỳ chọn) chế độ agent: nhận việc Tầng 1 từ worker máy bạn qua tunnel |

Hướng dẫn: `docs/kaggle-colab.md`, `docs/colab-dev-setup.md`; thuê GPU: `docs/gpu-rental.md`.

## 1. Nạp dữ liệu (tự tải, Drive hoặc upload)
Mặc định Colab **tự tải nuScenes Mini** từ link công khai của nuScenes (`SOURCE_URL`, ~3,9 GB, không cần upload, không cần tài khoản). Muốn dùng dữ liệu của riêng bạn: đặt `SOURCE_URL = ""` rồi chép `v1.0-mini.tgz` / `vcf_part_*.zip` vào `MyDrive/vcf_upload` (`DRIVE_DATA_DIR`), hoặc upload tay bằng **Choose Files**.

Thứ tự: đã giải nén → `SOURCE_URL` → file trong Drive → nút upload.

In [ ]:
# Ô 1 - NẠP DỮ LIỆU (Colab). Thứ tự: đã giải nén -> SOURCE_URL -> file trong Drive (DRIVE_DATA_DIR) -> nút upload.
import gc, glob, os, shutil, subprocess, tarfile, zipfile

USE_DRIVE = True        # True: mount Drive: đọc dữ liệu từ DRIVE_DATA_DIR + lưu kết quả/cache lên Drive.
                        # False: không dùng Drive (dữ liệu upload tay, mọi thứ mất khi phiên kết thúc).
DRIVE_DATA_DIR = "/content/drive/MyDrive/vcf_upload"   # thả v1.0-mini.tgz hoặc vcf_part_*.zip vào đây (hoặc thư mục nuScenes đã giải nén)
SOURCE_URL = "https://motional-nuscenes.s3.ap-northeast-1.amazonaws.com/public/v1.0/v1.0-mini.tgz"
                        # link công khai của nuScenes Mini (không cần đăng nhập, Colab tự tải ~vài phút).
                        # Đặt "" để dùng file trong Drive / upload tay.
DEST = "/content/nuscenes"   # giải nén vào ổ tạm của Colab (nhanh)


def _safe(root, name):
    out = os.path.realpath(os.path.join(root, name))
    assert out == root or out.startswith(root + os.sep), f"đường dẫn không an toàn: {name}"
    return out


def _find_nusc(base, depth=3):
    """Thư mục (base hoặc con trong `depth` cấp) chứa v1.0-*, hoặc None."""
    for d in range(depth + 1):
        hit = [h for h in glob.glob(os.path.join(base, *["*"] * d, "v1.0-*")) if os.path.isdir(h)]
        if hit:
            return os.path.dirname(sorted(hit)[0])
    return None


def _extract(path, root):
    n = 0
    if tarfile.is_tarfile(path):
        with tarfile.open(path) as tf:
            for m in tf:
                if not (m.isfile() or m.isdir()):
                    continue
                _safe(root, m.name)
                tf.extract(m, root)
                n += 1
    elif zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as zf:
            for m in zf.infolist():
                name = m.filename[5:] if m.filename.startswith("data/") else m.filename  # vcf-pack
                if m.is_dir() or not name:
                    continue
                out = _safe(root, name)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with zf.open(m) as s, open(out, "wb") as d:
                    shutil.copyfileobj(s, d)
                n += 1
    else:
        print("bỏ qua (không phải tar/zip):", path)
    return n


def _download(url, target):
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if "drive.google.com" in url:
        import gdown
        gdown.download(url, target, quiet=False, fuzzy=True)
    else:
        r = subprocess.run(["wget", "-q", "--show-progress", "-O", target, url])
        assert r.returncode == 0, "Tải thất bại: link nuscenes.org chỉ sống ít phút, hãy copy link mới"
    return target


def _unpack(files, dest):
    os.makedirs(dest, exist_ok=True)
    root = os.path.realpath(dest)
    total = sum(_extract(f, root) for f in files)
    found = _find_nusc(dest)
    assert found, f"Giải nén {total} file nhưng không thấy thư mục v1.0-* trong {dest}"
    print(f"Đã giải nén {total} file từ {len(files)} tệp")
    return found


if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

NUSC_READY = _find_nusc(DEST) or (_find_nusc(DRIVE_DATA_DIR) if USE_DRIVE else None)
if NUSC_READY:
    print("Đã có dữ liệu ở", NUSC_READY, "- bỏ qua")
elif SOURCE_URL.strip():
    NUSC_READY = _unpack([_download(SOURCE_URL.strip(), "/content/download/data.bin")], DEST)
elif USE_DRIVE and any(os.path.isfile(f) for f in glob.glob(DRIVE_DATA_DIR + "/*")):
    drive_files = sorted(f for f in glob.glob(DRIVE_DATA_DIR + "/*") if os.path.isfile(f))
    print("Dùng file trong Drive:", [os.path.basename(f) for f in drive_files])
    NUSC_READY = _unpack(drive_files, DEST)     # giải nén vào ổ tạm, KHÔNG xoá file trên Drive
else:
    from google.colab import files as colab_files
    print("Không thấy dữ liệu trong", DRIVE_DATA_DIR if USE_DRIVE else "Drive (USE_DRIVE=False)", "- upload tay.")
    print("Bấm 'Choose Files' bên dưới và chọn file nuScenes (v1.0-mini.tgz hoặc các vcf_part_*.zip).")
    print("File lớn (vài GB) upload khá lâu và cần RAM; đừng đóng tab cho tới khi xong.")
    up = colab_files.upload()
    paths = []
    for name, data in up.items():
        path = os.path.join("/content", os.path.basename(name))
        if not os.path.isfile(path):          # upload() thường đã ghi file; ghi bù nếu chưa
            with open(path, "wb") as f:
                f.write(data)
        paths.append(path)
    del up
    gc.collect()
    assert paths, "Chưa chọn file nào"
    NUSC_READY = _unpack(paths, DEST)
    for pth in paths:                         # giải phóng ổ đĩa sau khi giải nén
        os.remove(pth)
print("NUSC_READY =", NUSC_READY, "|", sorted(os.listdir(NUSC_READY)))


## 2. Cấu hình
`USE_DRIVE = True` (ô 1): Drive chứa dữ liệu đầu vào (`DRIVE_DATA_DIR`), kết quả `EXP` và cache bản build OpenPCDet, nên sống sót khi phiên bị ngắt. `False`: không dùng Drive. `GIT_REF` đổi thành `main` sau khi nhánh được merge.

In [ ]:
# Ô 2 - cấu hình (Colab)
import os, sys, glob, shutil, subprocess
from pathlib import Path

GIT_URL = "https://github.com/Nguyendev037/VCuboidFit.git"
GIT_REF = "fix/Linh"
NUSC = NUSC_READY
WORK = Path("/content")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")            # đã mount ở ô 1 thì bỏ qua; lưu kết quả + cache
    EXP = "/content/drive/MyDrive/vcf_exp/mini"
else:
    EXP = "/content/vcf_exp/mini"
Path(EXP).mkdir(parents=True, exist_ok=True)

REPO = WORK / "vcf"
if not (REPO / "model" / "scripts" / "tier1.sh").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, GIT_URL, str(REPO)], check=True)
REPO = str(REPO / "model")
os.environ.update(NUSC=NUSC, EXP=EXP, REPO=REPO)
print("NUSC =", NUSC, "\nEXP  =", EXP, "\nREPO =", REPO)


## 3. Kiểm GPU
Cần CUDA 12.x có sẵn trong image Kaggle/Colab. **P100 (sm_60) không nằm trong danh sách arch** ⇒ đổi sang T4 (Kaggle: Accelerator → *GPU T4 x2*).

In [ ]:
import torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "không có nvidia-smi")
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available(), "Không có GPU: bật Accelerator GPU rồi chạy lại (Tầng 0 chạy CPU được: bỏ qua phần 3, 4 và 6)."
CAP = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % CAP)
if CAP < (7, 0):
    raise SystemExit("GPU sm_%d%d (vd P100) KHÔNG được hỗ trợ — chọn T4/L4/A100." % CAP)
ARCH = "%d.%d" % CAP          # build CUDA ops đúng GPU đang chạy (T4=7.5, A100=8.0, L4=8.9)
os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
print("TORCH_CUDA_ARCH_LIST =", ARCH)

## 4. Cài môi trường (cache trên Drive nếu `USE_DRIVE`)
Dùng `scripts/colab_setup.sh`: cài thư viện, build OpenPCDet và **giữ bản build trên Drive** (khi `USE_DRIVE = True`) nên phiên sau không phải build lại (~10 phút). `numpy<2` có thể cần **Restart session**: Colab sẽ báo, restart rồi chạy lại từ ô 1.

In [ ]:
# Ô 4 - cài môi trường qua scripts/colab_setup.sh (idempotent, cache OpenPCDet trên Drive)
VCF_HOME = "/content/drive/MyDrive/vcf" if USE_DRIVE else "/content/vcf_home"
os.makedirs(VCF_HOME, exist_ok=True)
os.environ["VCF_HOME"] = VCF_HOME
r = subprocess.run(["bash", f"{REPO}/scripts/colab_setup.sh"], check=False)
assert r.returncode == 0, "colab_setup.sh lỗi: xem log phía trên"
PCDET_ROOT = f"{VCF_HOME}/OpenPCDet"
os.environ["PCDET_ROOT"] = PCDET_ROOT
os.environ["PYTHONPATH"] = f"{REPO}/worker:{PCDET_ROOT}:" + os.environ.get("PYTHONPATH", "")
for p in (f"{REPO}/worker", PCDET_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)
import importlib; importlib.invalidate_caches()
import pcdet, pcdet.datasets   # nếu lỗi np.int / av2: patch_pcdet chưa chạy
print("pcdet", pcdet.__version__, "| PCDET_ROOT =", PCDET_ROOT)


In [ ]:
# Smoke test (tuỳ chọn): bộ test LiDAR không cần GPU, để chắc môi trường ổn trước khi chạy thí nghiệm
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "not perf and not gpu", "tests/lidar"],
                   cwd=f"{REPO}/worker")
print("SMOKE OK" if r.returncode == 0 else "SMOKE FAIL: xem log pytest phía trên")


## 5. Tầng 0 (CPU): descriptor hình học, chọn 5%, tune
Tạo `EXP/index.parquet` + `features/` — **bắt buộc** trước Tầng 1 (`tier1.sh` thoát mã 4 nếu thiếu).

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

## 6. Tầng 1 (GPU): PointPillars huấn luyện từ seed S + suy luận
`--sweeps 1` (chỉ keyframe — ghi rõ trong báo cáo, dùng cùng giá trị cho downstream). Hết VRAM: `--batch 2` rồi `1`. Checkpoint ở `EXP/t1/ckpt/seed_latest.pth`; chạy lại thì phần đã có được bỏ qua.

In [ ]:
%%bash
set -e
bash "$REPO/scripts/tier1.sh" "$NUSC" "$EXP" --no-docker --sweeps 1 --epochs 20 --batch 4

## 7. Chạy lại Tầng 0 để có ma trận Tầng 1
Lần này có `t1/signals.parquet` ⇒ ma trận thêm `hybrid_mmr`, `t1_*_mmr`, `entropy_only`.

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

In [ ]:
from IPython.display import Markdown, display
rep = Path(EXP) / "V" / "report_selection.md"
display(Markdown(rep.read_text(encoding="utf-8")) if rep.exists() else Markdown(f"Chưa có `{rep}`"))

## 8. Lưu kết quả
Zip `EXP` **không kèm** `t1/pcdet_data` (chỉ symlink + info tạm, tái tạo được) và đặt cạnh `EXP` trên Drive.

**Resume:** phiên ngắt ⇒ mở lại, chạy phần 1 → 4, rồi chạy tiếp phần 5–7; các bước đã xong tự bỏ qua.

In [ ]:
import zipfile, time
OUT_DIR = Path("/content/drive/MyDrive") if USE_DRIVE else Path("/content")
zpath = OUT_DIR / f"vcf_exp_{time.strftime('%Y%m%d_%H%M')}.zip"
skip = Path(EXP) / "t1" / "pcdet_data"
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(EXP, followlinks=False):
        dirs[:] = [d for d in dirs if Path(root, d) != skip]
        for f in files:
            fp = Path(root, f)
            if fp.is_symlink():
                continue
            z.write(fp, fp.relative_to(Path(EXP).parent)); n += 1
print(f"{zpath}  {zpath.stat().st_size/1e6:.1f} MB, {n} file")

## 9. (Tuỳ chọn) Chế độ agent: nhận việc Tầng 1 từ worker máy bạn
Dùng khi máy chạy worker không có GPU. Cần URL THẬT do `cloudflared` in ra trên máy đó và token `VCF_REMOTE_TOKEN` của worker. Chạy xong các ô 1-4 là đủ, không cần Tầng 0/1 ở trên. Hướng dẫn: `docs/colab-dev-setup.md` mục 4.

In [ ]:
# Ô 9 - agent: kéo việc Tầng 1 từ worker, train + suy luận trên GPU Colab, đẩy signals.parquet về
import urllib.request
from getpass import getpass

SERVER = input("URL tunnel của worker (https://<tên>.trycloudflare.com): ").strip().rstrip("/")
if not SERVER.startswith("https://") or "xxxx" in SERVER:
    raise SystemExit("URL không hợp lệ: dán URL THẬT do cloudflared in ra trên máy chạy worker.")
try:
    print("Kiểm /health:", urllib.request.urlopen(SERVER + "/health", timeout=20).read().decode()[:120])
except Exception as e:
    raise SystemExit(f"Không gọi được {SERVER}/health ({e}). Kiểm cloudflared còn chạy, đợi 30 s rồi chạy lại.")
os.environ["VCF_REMOTE_TOKEN"] = getpass("VCF_REMOTE_TOKEN (không hiện ký tự): ")
subprocess.run([sys.executable, "-u", f"{REPO}/scripts/colab_agent.py", "--server", SERVER,
                "--work", "/content/vcf_agent_work"], env=os.environ)


## 10. (Tuỳ chọn, mặc định tắt) Cổng G1
Đặt `RUN_G1 = True` để chạy `c4.cli.lidar_g1`. Chỉ bật khi đã đọc SPEC của cổng G1.

In [ ]:
RUN_G1 = False
if RUN_G1:
    r = subprocess.run([sys.executable, "-m", "c4.cli.lidar_g1", "--data-root", NUSC, "--out", EXP],
                       cwd=f"{REPO}/worker")
    print("exit", r.returncode)
else:
    print("G1 tắt (RUN_G1 = False)")